# Mini-Project – Marketing Strategy Analysis
**Dataset :** US Superstore Data  
**Objectif :** Analyser les ventes par zone géographique, par client et par produit pour définir une stratégie marketing.  
**Outils :** Pandas, NumPy, Matplotlib, Seaborn

## 1. Chargement et préparation des données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

In [ ]:
df = pd.read_excel('US Superstore data.xls')

print('Dimensions :', df.shape)
print('Colonnes   :', df.columns.tolist())
df.head()

In [ ]:
# Vérifications de base
print('Valeurs manquantes :')
print(df.isnull().sum())
print(f'\nDoublons : {df.duplicated().sum()}')

# S'assurer que les dates sont bien en datetime
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])

# Créer des colonnes utiles
df['Order Year']  = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month

print('\nTypes des colonnes :')
print(df.dtypes)

## Q1 – Quels états ont le plus de ventes ?

In [ ]:
sales_by_state = df.groupby('State')['Sales'].sum().sort_values(ascending=False)
top10_states   = sales_by_state.head(10)

plt.figure(figsize=(13, 5))
sns.barplot(x=top10_states.values, y=top10_states.index, palette='Blues_r')
plt.title('Top 10 États par Ventes Totales', fontsize=13, fontweight='bold')
plt.xlabel('Ventes ($)')
plt.ylabel('')

for i, val in enumerate(top10_states.values):
    plt.text(val + 1000, i, f'${val:,.0f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print('Top 5 états :')
print(top10_states.head())

## Q2 – Différence entre New York et la Californie (ventes et profit)

In [ ]:
ny_ca = df[df['State'].isin(['New York', 'California'])]
comparison = ny_ca.groupby('State')[['Sales', 'Profit']].sum().reset_index()

print('Comparaison New York vs California :')
print(comparison.to_string(index=False))

# Graphique côte à côte
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = ['#3498db', '#e74c3c']

axes[0].bar(comparison['State'], comparison['Sales'], color=colors, edgecolor='white')
axes[0].set_title('Ventes totales : NY vs CA', fontweight='bold')
axes[0].set_ylabel('Ventes ($)')
for i, val in enumerate(comparison['Sales']):
    axes[0].text(i, val + 1000, f'${val:,.0f}', ha='center', fontsize=10)

axes[1].bar(comparison['State'], comparison['Profit'], color=colors, edgecolor='white')
axes[1].set_title('Profit total : NY vs CA', fontweight='bold')
axes[1].set_ylabel('Profit ($)')
for i, val in enumerate(comparison['Profit']):
    axes[1].text(i, val + 500, f'${val:,.0f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## Q3 – Quel est le meilleur client à New York ?

In [ ]:
ny_customers = df[df['State'] == 'New York'].groupby('Customer Name')['Sales'].sum()
ny_customers = ny_customers.sort_values(ascending=False)
top_ny = ny_customers.head(10)

print(f'Meilleur client à New York : {ny_customers.index[0]} (${ny_customers.iloc[0]:,.0f})')

plt.figure(figsize=(12, 5))
sns.barplot(x=top_ny.values, y=top_ny.index, palette='Greens_r')
plt.title('Top 10 Clients à New York par Ventes', fontsize=13, fontweight='bold')
plt.xlabel('Ventes ($)')
plt.ylabel('')
plt.tight_layout()
plt.show()

## Q4 – Y a-t-il des différences de rentabilité entre états ?

In [ ]:
profit_by_state  = df.groupby('State')['Profit'].sum().sort_values()
profit_margin_by = (df.groupby('State')['Profit'].sum() / df.groupby('State')['Sales'].sum() * 100)

# Afficher les états les plus et moins rentables
print('5 états les MOINS rentables :')
print(profit_by_state.head(5))
print('\n5 états les PLUS rentables :')
print(profit_by_state.tail(5))

# Graphique : tous les états, profit total
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in profit_by_state.values]
axes[0].barh(profit_by_state.index, profit_by_state.values, color=colors)
axes[0].axvline(x=0, color='black', linewidth=0.8)
axes[0].set_title('Profit total par État', fontweight='bold')
axes[0].set_xlabel('Profit ($)')
axes[0].tick_params(axis='y', labelsize=7)

# Top 10 et bottom 10 par marge
margin_sorted = profit_margin_by.sort_values()
bottom5 = margin_sorted.head(5)
top5    = margin_sorted.tail(5)
combined = pd.concat([bottom5, top5])
bar_colors = ['#e74c3c'] * 5 + ['#2ecc71'] * 5
axes[1].barh(combined.index, combined.values, color=bar_colors)
axes[1].axvline(x=0, color='black', linewidth=0.8)
axes[1].set_title('Marge (%) : 5 pires vs 5 meilleurs états', fontweight='bold')
axes[1].set_xlabel('Marge bénéficiaire (%)')

plt.tight_layout()
plt.show()

## Q5 – Principe de Pareto : 20% des clients → 80% du profit ?

In [ ]:
# Calculer le profit par client, trié décroissant
profit_by_customer = df.groupby('Customer Name')['Profit'].sum().sort_values(ascending=False)

# Calcul des pourcentages cumulés
nb_customers      = len(profit_by_customer)
cumulative_profit = profit_by_customer.cumsum()
total_profit      = profit_by_customer.sum()

cumulative_profit_pct  = cumulative_profit / total_profit * 100
cumulative_customer_pct = np.arange(1, nb_customers + 1) / nb_customers * 100

# Trouver le seuil des 80% de profit
idx_80 = (cumulative_profit_pct >= 80).idxmax()
rank_80 = profit_by_customer.index.get_loc(idx_80) + 1
pct_customers_80 = rank_80 / nb_customers * 100

print(f'Total clients         : {nb_customers}')
print(f'Clients pour 80% du profit : {rank_80} ({pct_customers_80:.1f}%)')
print(f'→ La règle 80/20 s\'applique : {pct_customers_80 <= 25}')

# Courbe de Pareto
plt.figure(figsize=(12, 5))
plt.plot(cumulative_customer_pct, cumulative_profit_pct, color='steelblue', linewidth=2)
plt.axhline(y=80, color='red',  linestyle='--', alpha=0.7, label='80% du profit')
plt.axvline(x=pct_customers_80, color='orange', linestyle='--', alpha=0.7,
            label=f'{pct_customers_80:.1f}% des clients')
plt.scatter([pct_customers_80], [80], color='red', s=80, zorder=5)

plt.title('Courbe de Pareto – Clients vs Profit cumulé', fontsize=13, fontweight='bold')
plt.xlabel('% Clients cumulés')
plt.ylabel('% Profit cumulé')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Q6 – Top 20 villes par Ventes et par Profit

In [ ]:
sales_by_city  = df.groupby('City')['Sales'].sum().sort_values(ascending=False).head(20)
profit_by_city = df.groupby('City')['Profit'].sum().sort_values(ascending=False).head(20)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Top 20 villes par ventes
axes[0].barh(sales_by_city.index[::-1], sales_by_city.values[::-1], color='steelblue')
axes[0].set_title('Top 20 Villes par Ventes', fontweight='bold')
axes[0].set_xlabel('Ventes ($)')

# Top 20 villes par profit
colors_profit = ['#e74c3c' if v < 0 else '#2ecc71' for v in profit_by_city.values[::-1]]
axes[1].barh(profit_by_city.index[::-1], profit_by_city.values[::-1], color=colors_profit)
axes[1].set_title('Top 20 Villes par Profit', fontweight='bold')
axes[1].set_xlabel('Profit ($)')

plt.tight_layout()
plt.show()

# Comparer les villes dans les deux classements
top20_sales_cities   = set(sales_by_city.index)
top20_profit_cities  = set(profit_by_city.index)
cities_in_both       = top20_sales_cities & top20_profit_cities
print(f'Villes dans les deux top 20 : {len(cities_in_both)}')
print(cities_in_both)

## Q7 – Top 20 clients par Ventes

In [ ]:
top20_customers = df.groupby('Customer Name')['Sales'].sum().sort_values(ascending=False).head(20)

plt.figure(figsize=(13, 7))
sns.barplot(x=top20_customers.values, y=top20_customers.index, palette='Oranges_r')
plt.title('Top 20 Clients par Ventes Totales', fontsize=13, fontweight='bold')
plt.xlabel('Ventes ($)')
plt.ylabel('')

for i, val in enumerate(top20_customers.values):
    plt.text(val + 200, i, f'${val:,.0f}', va='center', fontsize=8)

plt.tight_layout()
plt.show()

## Q8 – Courbe cumulative des ventes par client (Pareto)

In [ ]:
sales_by_customer = df.groupby('Customer Name')['Sales'].sum().sort_values(ascending=False)

nb_cust           = len(sales_by_customer)
cumul_sales       = sales_by_customer.cumsum()
total_sales       = sales_by_customer.sum()

cumul_sales_pct   = cumul_sales / total_sales * 100
cumul_cust_pct    = np.arange(1, nb_cust + 1) / nb_cust * 100

# Trouver le seuil des 80% des ventes
idx_80_sales     = (cumul_sales_pct >= 80).idxmax()
rank_80_sales    = sales_by_customer.index.get_loc(idx_80_sales) + 1
pct_cust_80sales = rank_80_sales / nb_cust * 100

print(f'Clients pour 80% des ventes : {rank_80_sales} ({pct_cust_80sales:.1f}%)')

plt.figure(figsize=(12, 5))
plt.plot(cumul_cust_pct, cumul_sales_pct, color='coral', linewidth=2)
plt.axhline(y=80, color='red',    linestyle='--', alpha=0.7, label='80% des ventes')
plt.axvline(x=pct_cust_80sales, color='blue', linestyle='--', alpha=0.7,
            label=f'{pct_cust_80sales:.1f}% des clients')
plt.scatter([pct_cust_80sales], [80], color='red', s=80, zorder=5)

plt.title('Courbe de Pareto – Clients vs Ventes cumulées', fontsize=13, fontweight='bold')
plt.xlabel('% Clients cumulés')
plt.ylabel('% Ventes cumulées')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Q9 – Recommandations Marketing

In [ ]:
# Résumé des états prioritaires
state_summary = df.groupby('State').agg(
    total_sales   = ('Sales',  'sum'),
    total_profit  = ('Profit', 'sum'),
    nb_orders     = ('Order ID', 'nunique')
).reset_index()

state_summary['profit_margin'] = (state_summary['total_profit'] / state_summary['total_sales'] * 100).round(2)

# États avec bonnes ventes ET bon profit
good_states = state_summary[
    (state_summary['total_sales']  > state_summary['total_sales'].quantile(0.75)) &
    (state_summary['total_profit'] > 0)
].sort_values('total_profit', ascending=False)

# États à éviter (perte malgré du volume)
loss_states = state_summary[
    (state_summary['total_profit'] < 0)
].sort_values('total_profit')

print('=== RECOMMANDATIONS MARKETING ===')
print()
print('ÉTATS PRIORITAIRES (hautes ventes + profit positif) :')
print(good_states[['State', 'total_sales', 'total_profit', 'profit_margin']].head(8).to_string(index=False))
print()
print('ÉTATS À REVOIR (pertes à corriger) :')
print(loss_states[['State', 'total_sales', 'total_profit']].head(5).to_string(index=False))

# Visualisation finale
plt.figure(figsize=(12, 6))
plt.scatter(
    state_summary['total_sales'],
    state_summary['total_profit'],
    c=state_summary['profit_margin'],
    cmap='RdYlGn',
    s=80,
    alpha=0.8
)
plt.colorbar(label='Marge (%)')
plt.axhline(y=0, color='black', linestyle='--', alpha=0.5)

# Annoter les états importants
for _, row in state_summary.nlargest(5, 'total_sales').iterrows():
    plt.annotate(row['State'], (row['total_sales'], row['total_profit']),
                 fontsize=8, xytext=(5, 5), textcoords='offset points')
for _, row in state_summary.nsmallest(3, 'total_profit').iterrows():
    plt.annotate(row['State'], (row['total_sales'], row['total_profit']),
                 fontsize=8, color='red', xytext=(5, -12), textcoords='offset points')

plt.title('Ventes vs Profit par État (couleur = marge)', fontsize=13, fontweight='bold')
plt.xlabel('Ventes totales ($)')
plt.ylabel('Profit total ($)')
plt.tight_layout()
plt.show()

## Conclusion et Stratégie Marketing

**États à prioriser :**  
La Californie et New York dominent les ventes. New York a une meilleure rentabilité relative. Ces deux états méritent des campagnes marketing soutenues et une attention particulière aux clients VIP identifiés.

**Principe de Pareto :**  
Une minorité de clients génère l'essentiel des ventes et du profit. Il est stratégiquement efficace de cibler et fidéliser ces clients à forte valeur plutôt que de disperser les efforts sur l'ensemble du portefeuille.

**Villes clés :**  
New York City, Los Angeles et Seattle sont en tête des ventes. Cependant, certaines villes avec de bonnes ventes affichent des profits faibles ou négatifs — signe de remises excessives ou de coûts logistiques élevés à investiguer.

**Action recommandée :**  
1. Concentrer 80% des efforts marketing sur les états et villes du top quartile (ventes + profit positif)
2. Réduire ou supprimer les remises dans les zones déficitaires
3. Créer un programme de fidélité ciblant les 20% de clients générant 80% du profit